# Cisco CX CDC: PySpark Hash-Based CDC (Portable)

**Source**: S3 Parquet (10M rows)  
**CDC**: MD5 hash + full outer join in Spark (reads parquet directly - no pandas conversion)  
**Target**: Snowflake table  
**Runs on**: Snowflake SPCS and EC2/Docker identically

In [1]:
import importlib, os, subprocess, sys

if not os.path.exists('/usr/lib/jvm'):
    try:
        subprocess.run(['apt-get', 'update', '-qq'], capture_output=True, timeout=60)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'default-jdk'], capture_output=True, timeout=120)
        print('Java installed.')
    except Exception as e:
        print(f'Java install skipped: {e}')

for jvm_path in ['/usr/lib/jvm/java-11-openjdk-amd64', '/usr/lib/jvm/java-17-openjdk-amd64',
                 '/usr/lib/jvm/default-java']:
    if os.path.exists(jvm_path):
        os.environ['JAVA_HOME'] = jvm_path
        break

# Use PySpark 3.5.3 (same as EC2) — Spark 4.x has a known Hadoop 3.4 S3A bug
# that causes NumberFormatException: "60s"
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyspark==3.5.3', 'pyarrow', '-q'])
print('pyspark 3.5.3 installed (matching EC2 setup).')

# Set PYSPARK_SUBMIT_ARGS to pull in hadoop-aws on JVM startup (same as EC2)
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.hadoop:hadoop-aws:3.3.4 pyspark-shell'

print(f"JAVA_HOME={os.environ.get('JAVA_HOME', 'NOT SET')}")
print(f"PYSPARK_SUBMIT_ARGS={os.environ['PYSPARK_SUBMIT_ARGS']}")

pyspark 3.5.3 installed (matching EC2 setup).
JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
PYSPARK_SUBMIT_ARGS=--packages org.apache.hadoop:hadoop-aws:3.3.4 pyspark-shell


In [2]:
import os

def get_snowflake_connection():
    """Connect to Snowflake - works in SPCS or on-prem."""
    try:
        from snowflake.snowpark.context import get_active_session
        session = get_active_session()
        print('Connected via Snowflake SPCS')
        return session, 'SPCS'
    except:
        pass
    import snowflake.connector
    conn = snowflake.connector.connect(connection_name=os.getenv('SNOWFLAKE_CONNECTION_NAME', 'default'))
    print(f'Connected locally to {conn.account}')
    return conn, 'LOCAL'

In [3]:
import time
import os
import json

S3_BUCKET = 'cisco-cx-cdc-pilot'
COMPARE_COLS = ['software_version', 'cpu_utilization', 'memory_utilization',
                'critical_bugs_count', 'contract_status', 'ip_address']

# Load AWS credentials from Snowflake credential store table
# Update creds with: UPDATE CISCO_CX_PILOT.SECRETS.AWS_CREDS SET creds = '{...}' WHERE id = 1;
conn, runtime = get_snowflake_connection()
if runtime == 'LOCAL':
    cur = conn.cursor()
    cur.execute("SELECT creds FROM CISCO_CX_PILOT.SECRETS.AWS_CREDS WHERE id = 1")
    creds_json = cur.fetchone()[0]
else:
    session = conn
    result = session.sql("SELECT creds FROM CISCO_CX_PILOT.SECRETS.AWS_CREDS WHERE id = 1").collect()
    creds_json = result[0][0]

creds = json.loads(creds_json)
os.environ['AWS_ACCESS_KEY_ID'] = creds['aws_access_key_id']
os.environ['AWS_SECRET_ACCESS_KEY'] = creds['aws_secret_access_key']
os.environ['AWS_SESSION_TOKEN'] = creds.get('aws_session_token', '')

# READ_MODE options:
#   'S3_DIRECT' - Read parquet from S3 via Spark S3A (uses AWS keys from credential table)
#   'STAGE'     - Read via Snowflake external stage + pyarrow (original SPCS path)
READ_MODE = 'S3_DIRECT'

def detect_runtime():
    try:
        from snowflake.snowpark.context import get_active_session
        get_active_session()
        return 'SPCS'
    except:
        return 'LOCAL'

RUNTIME = detect_runtime()
print(f'Runtime: {RUNTIME} | Read mode: {READ_MODE}')
print(f'Source: s3://{S3_BUCKET}/cdc_data/')
print('AWS credentials loaded from Snowflake credential store.')

Connected locally to FZB62295
Runtime: LOCAL | Read mode: S3_DIRECT
Source: s3://cisco-cx-cdc-pilot/cdc_data/
AWS credentials loaded from Snowflake credential store.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import md5, concat_ws, col, lit, when, broadcast
import time

# Stop any prior Spark session
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName('CiscoCX_CDC') \
    .config('spark.driver.memory', '64g') \
    .config('spark.sql.shuffle.partitions', '400') \
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true') \
    .master('local[*]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

# Override problematic Hadoop 3.4.0 S3A configs that use "60s" string format
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
hadoop_conf.set('fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID'])
hadoop_conf.set('fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY'])
if os.environ.get('AWS_SESSION_TOKEN'):
    hadoop_conf.set('fs.s3a.session.token', os.environ['AWS_SESSION_TOKEN'])
    hadoop_conf.set('fs.s3a.aws.credentials.provider',
                    'org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider')
else:
    hadoop_conf.set('fs.s3a.aws.credentials.provider',
                    'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')

# Fix Hadoop 3.4.0 NumberFormatException: "60s" -- override all duration configs with ms values
hadoop_conf.set('fs.s3a.connection.timeout', '60000')
hadoop_conf.set('fs.s3a.connection.establish.timeout', '60000')
hadoop_conf.set('fs.s3a.attempts.maximum', '10')
hadoop_conf.set('fs.s3a.retry.limit', '10')
hadoop_conf.set('fs.s3a.retry.interval', '500')
hadoop_conf.set('fs.s3a.retry.throttle.limit', '10')
hadoop_conf.set('fs.s3a.retry.throttle.interval', '1000')
hadoop_conf.set('fs.s3a.connection.maximum', '96')
hadoop_conf.set('fs.s3a.threads.max', '64')
hadoop_conf.set('fs.s3a.socket.send.buffer', '8192')
hadoop_conf.set('fs.s3a.socket.recv.buffer', '8192')
hadoop_conf.set('fs.s3a.readahead.range', '65536')
hadoop_conf.set('fs.s3a.multipart.size', '67108864')
hadoop_conf.set('fs.s3a.multipart.threshold', '134217728')
hadoop_conf.set('fs.s3a.max.total.tasks', '32')
hadoop_conf.set('fs.s3a.bulk.delete.page.size', '250')

print(f'Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}')

# --- PREP: Read prev_snapshot from S3 and load into Snowflake ---
t0 = time.time()
print('Reading prev_snapshot from S3...')
prev_s3_spark = spark.read.parquet(f's3a://{S3_BUCKET}/cdc_data/prev_snapshot/')
prev_count = prev_s3_spark.count()
t_read_prev_s3 = time.time() - t0
print(f'  prev_snapshot read from S3: {prev_count:,} rows in {t_read_prev_s3:.1f}s')

# Load prev_snapshot into Snowflake
t0 = time.time()
conn, runtime = get_snowflake_connection()
prev_pdf = prev_s3_spark.toPandas()
prev_pdf.columns = [c.upper() for c in prev_pdf.columns]

TARGET_DB = 'CISCO_CX_PILOT'
TARGET_SCHEMA = 'PROCESSED'
PREV_TABLE = 'PREV_SNAPSHOT_STAGING'

if runtime == 'LOCAL':
    from snowflake.connector.pandas_tools import write_pandas
    conn.cursor().execute(f'USE DATABASE {TARGET_DB}')
    conn.cursor().execute(f'USE SCHEMA {TARGET_SCHEMA}')
    conn.cursor().execute(f"""CREATE TABLE IF NOT EXISTS {PREV_TABLE} (
        DEVICE_ID VARCHAR, CUSTOMER_ID VARCHAR, SOFTWARE_VERSION VARCHAR,
        CPU_UTILIZATION FLOAT, MEMORY_UTILIZATION FLOAT, CRITICAL_BUGS_COUNT INTEGER,
        CONTRACT_STATUS VARCHAR, IP_ADDRESS VARCHAR, PRODUCT_FAMILY VARCHAR)""")
    conn.cursor().execute(f'TRUNCATE TABLE {PREV_TABLE}')
    success, nchunks, nrows, _ = write_pandas(conn, prev_pdf, PREV_TABLE, database=TARGET_DB, schema=TARGET_SCHEMA)
else:
    session = conn
    session.sql(f'USE DATABASE {TARGET_DB}').collect()
    session.sql(f'USE SCHEMA {TARGET_SCHEMA}').collect()
    session.write_pandas(prev_pdf, PREV_TABLE, database=TARGET_DB, schema=TARGET_SCHEMA, overwrite=True)

t_load_sf = time.time() - t0
print(f'  prev_snapshot loaded into Snowflake ({TARGET_DB}.{TARGET_SCHEMA}.{PREV_TABLE}): {t_load_sf:.1f}s')
print(f'\n--- Prep complete (not included in pipeline time) ---')

:: loading settings :: url = jar:file:/home/ubuntu/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-80083233-9d3c-406b-8882-c6df714ea43b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central


	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 222ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-80083233-9d3c-406b-8882-c6df714ea43b
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/5ms)


26/06/18 19:58:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.3 | Cores: 16
Reading prev_snapshot from S3...


26/06/18 19:58:26 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


  prev_snapshot read from S3: 10,000,000 rows in 6.6s


Connected locally to FZB62295


  prev_snapshot loaded into Snowflake (CISCO_CX_PILOT.PROCESSED.PREV_SNAPSHOT_STAGING): 150.3s

--- Prep complete (not included in pipeline time) ---


## CDC: Spark Hash Diff + Full Outer Join

MD5 hash of compare columns, full outer join on device_id, classify INSERT/UPDATE/DELETE

In [5]:
# --- PIPELINE START: Read prev from Snowflake + Read curr from S3 + CDC ---
t_pipeline_start = time.time()

# Read prev_snapshot back from Snowflake
t0 = time.time()
if runtime == 'LOCAL':
    import pandas as pd
    cur = conn.cursor()
    cur.execute(f'SELECT * FROM {TARGET_DB}.{TARGET_SCHEMA}.{PREV_TABLE}')
    prev_from_sf = cur.fetch_pandas_all()
else:
    prev_from_sf = session.sql(f'SELECT * FROM {TARGET_DB}.{TARGET_SCHEMA}.{PREV_TABLE}').to_pandas()

prev_from_sf.columns = [c.lower() for c in prev_from_sf.columns]
prev_spark = spark.createDataFrame(prev_from_sf)
t_read_sf = time.time() - t0
print(f'  prev_snapshot read from Snowflake into Spark: {t_read_sf:.1f}s')

# Read curr_snapshot from S3
t0 = time.time()
print('Reading curr_snapshot from S3...')
curr_spark = spark.read.parquet(f's3a://{S3_BUCKET}/cdc_data/curr_snapshot/')
curr_count = curr_spark.count()
t_read_curr_s3 = time.time() - t0
print(f'  curr_snapshot read from S3: {curr_count:,} rows in {t_read_curr_s3:.1f}s')

print(f'\n--- Data Ready for CDC ---')
print(f'  Previous (from Snowflake): {prev_count:,}')
print(f'  Current  (from S3):        {curr_count:,}')

# --- Hash-based CDC ---
t0 = time.time()
prev_hashed = prev_spark.withColumn('_hash', md5(concat_ws('|', *[col(c).cast('string') for c in COMPARE_COLS])))
curr_hashed = curr_spark.withColumn('_hash', md5(concat_ws('|', *[col(c).cast('string') for c in COMPARE_COLS])))

prev_keys = prev_hashed.select(col('device_id'), col('_hash').alias('_hash_prev'))
curr_keys = curr_hashed.select(col('device_id'), col('_hash').alias('_hash_curr'))
t_hash = time.time() - t0

t0 = time.time()
merged = curr_keys.join(prev_keys, on='device_id', how='full')
merged = merged.withColumn('cdc_op',
    when(col('_hash_prev').isNull(), lit('INSERT'))
    .when(col('_hash_curr').isNull(), lit('DELETE'))
    .when(col('_hash_curr') != col('_hash_prev'), lit('UPDATE'))
    .otherwise(lit('NONE'))
)

changes = merged.filter(col('cdc_op') != 'NONE').cache()
n_inserts = changes.filter(col('cdc_op') == 'INSERT').count()
n_updates = changes.filter(col('cdc_op') == 'UPDATE').count()
n_deletes = changes.filter(col('cdc_op') == 'DELETE').count()
t_join = time.time() - t0

print(f'\n=== PySpark Hash-Based CDC Results ===')
print(f'  Hash compute (lazy):       {t_hash:.1f}s')
print(f'  Join + Classify:           {t_join:.1f}s')
print(f'  Inserts: {n_inserts:,} | Updates: {n_updates:,} | Deletes: {n_deletes:,}')

  prev_snapshot read from Snowflake into Spark: 27.7s
Reading curr_snapshot from S3...


  curr_snapshot read from S3: 10,020,000 rows in 1.0s

--- Data Ready for CDC ---
  Previous (from Snowflake): 10,000,000
  Current  (from S3):        10,020,000



=== PySpark Hash-Based CDC Results ===
  Hash compute (lazy):       0.1s
  Join + Classify:           15.2s
  Inserts: 25,000 | Updates: 18,952 | Deletes: 5,000


In [6]:
t0 = time.time()
# Reuse Snowflake connection from c04

upsert_ids = changes.filter(col('cdc_op').isin('INSERT', 'UPDATE')).select('device_id')
upsert_df = curr_hashed.join(broadcast(upsert_ids), on='device_id').drop('_hash').toPandas()
upsert_df.columns = [c.upper() for c in upsert_df.columns]

if runtime == 'LOCAL':
    from snowflake.connector.pandas_tools import write_pandas
    conn.cursor().execute('USE DATABASE CISCO_CX_PILOT')
    conn.cursor().execute('USE SCHEMA PROCESSED')
    conn.cursor().execute('CREATE TABLE IF NOT EXISTS SPARK_CDC_RESULT (DEVICE_ID VARCHAR, CUSTOMER_ID VARCHAR, SOFTWARE_VERSION VARCHAR, CPU_UTILIZATION FLOAT, MEMORY_UTILIZATION FLOAT, CRITICAL_BUGS_COUNT INTEGER, CONTRACT_STATUS VARCHAR, IP_ADDRESS VARCHAR, PRODUCT_FAMILY VARCHAR)')
    conn.cursor().execute('TRUNCATE TABLE SPARK_CDC_RESULT')
    success, nchunks, nrows, _ = write_pandas(conn, upsert_df, 'SPARK_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED')
    print(f'Wrote {nrows:,} rows to Snowflake in {time.time()-t0:.1f}s')
else:
    session = conn
    session.sql('USE DATABASE CISCO_CX_PILOT').collect()
    session.sql('USE SCHEMA PROCESSED').collect()
    session.write_pandas(upsert_df, 'SPARK_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED', overwrite=True)
    print(f'Wrote {len(upsert_df):,} rows to Snowflake in {time.time()-t0:.1f}s')

t_write = time.time() - t0
t_total = time.time() - t_pipeline_start
spark.stop()
print(f'\n=== TOTAL PIPELINE SUMMARY ===')
print(f'  Read prev from Snowflake:  {t_read_sf:.1f}s')
print(f'  Read curr from S3:         {t_read_curr_s3:.1f}s')
print(f'  Hash + Join + Classify:    {t_hash + t_join:.1f}s')
print(f'  Write results to SF:       {t_write:.1f}s')
print(f'  TOTAL PIPELINE:            {t_total:.1f}s')

Wrote 43,952 rows to Snowflake in 9.6s



=== TOTAL PIPELINE SUMMARY ===
  Read prev from Snowflake:  27.7s
  Read curr from S3:         1.0s
  Hash + Join + Classify:    15.4s
  Write results to SF:       9.6s
  TOTAL PIPELINE:            53.6s
